# Exploratory analysis

Real 2024 measurements from the SPARK dataset (KIT) and SMARD. Run `python run_pipeline.py` first so that `data/processed` exists.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import config as C
from src.feature_engineering import PQ_CLUSTER_FEATURES
df = pd.read_parquet(C.PROCESSED / 'analysis_dataset_10min.parquet')
print(df.shape)
print(list(df.columns))

## Coverage and basic statistics

In [ ]:
print(df.index.min(), df.index.max())
print(df.groupby(df.index.month).size())
df[PQ_CLUSTER_FEATURES].describe().T.round(3)

## Distributions of the clustering features

In [ ]:
ax = df[PQ_CLUSTER_FEATURES].hist(bins=60, figsize=(14, 8)); plt.tight_layout()

## Daily rhythm: power quality, facility load and regional solar

In [ ]:
h = df.groupby('hour_local')[['thd_u_pct','v_dev_pct','p_kw','solar_penetration_pct','wind_penetration_pct']].median()
h.plot(subplots=True, figsize=(9, 9), marker='o'); plt.tight_layout()

## Renewable penetration and level thresholds

In [ ]:
print(df.groupby('re_level', observed=True)['vre_penetration_pct'].agg(['min','max','size']))
df['vre_penetration_pct'].hist(bins=80, figsize=(8,3))

## Feature correlation (Spearman)

In [ ]:
df[PQ_CLUSTER_FEATURES].corr(method='spearman').round(2)

## k-distance curve used for the stage-1 epsilon

In [ ]:
from src.dbscan_model import prepare_matrix, k_distance, knee_point
pm = prepare_matrix(df, PQ_CLUSTER_FEATURES)
kd = k_distance(pm.X, 2 * pm.X.shape[1])
i, eps = knee_point(kd)
plt.figure(figsize=(7,3)); plt.plot(kd); plt.axhline(eps, ls='--', c='r'); plt.title(f'knee eps = {eps:.3f}')
print(eps, i / len(kd))

## Experiment labels (if `src.experiments` has run)

In [ ]:
p = C.RESULTS / 'exp1_labels.parquet'
if p.exists():
    lab = pd.read_parquet(p)
    print(lab.label_name.value_counts())
    d = df.join(lab['label'])
    print(d.groupby('re_level', observed=True)['label'].apply(lambda s: (s != 0).mean() * 100).round(2))